# Notebook 01 · Datos, preprocesamiento y modelo baseline (TensorFlow + Keras)

| Redes Neuronales III | 604013/DEEP LEARNING -CONCEPTOS/FU/CAD2202023205/EIAIIPA2026 |
|---|---|
| Universidad de Cundinamarca · Especialización en Inteligencia Artificial | Docente: **Nidia Stella Garcia Roa** |
| Estudiante: **Yohann Javier Nikolay Salazar Cardenas** | Repositorio: `https://github.com/Yohann0011/Redes-Neuronales-III` |


**Problema:** clasificación multiclase de dígitos MNIST (10 clases) con un perceptrón multicapa
784 → 128 → 10, **sin redes convolucionales**.

> **Alcance de este notebook:** división estratificada **Train / Dev / Test**, normalización de
> píxeles, revisión de la distribución de clases, construcción del modelo baseline y entrenamiento
> con **Dev explícito** como conjunto de validación.

---

## 1. Objetivos

1. Separar los datos en tres conjuntos con funciones distintas: **Train** aprende, **Dev** compara
   configuraciones y **Test** se reserva para la evaluación final.
2. Normalizar los píxeles a [0, 1] y verificar la distribución de clases con partición estratificada.
3. Construir y entrenar un modelo baseline cuya arquitectura sirva de punto de comparación para los
   experimentos de optimización y regularización de los notebooks siguientes.
4. Leer curvas de pérdida y exactitud para detectar señales tempranas de overfitting.

## 2. Por qué tres conjuntos y no dos

| Conjunto | Uso | Regla práctica |
|---|---|---|
| **Train** | Actualizar los pesos | El modelo puede verlo siempre |
| **Dev (validation)** | Comparar optimizadores, learning rates y regularizaciones | No actualiza pesos; guía decisiones |
| **Test** | Evaluación final única | No se toca durante el diseño: usarlo para decidir es *data leakage* |

Si las configuraciones se eligieran mirando Test, el reporte final dejaría de ser una estimación
honesta de la generalización: sería como estudiar con las preguntas del examen.

## 3. Entorno de ejecución

Diseñado para **Google Colab** (`Entorno de ejecución → Ejecutar todo`). Semilla fija en 42 para
reproducibilidad.

In [1]:
import os
# Ambiente silencioso y determinista para TensorFlow (no altera los resultados)
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "3"
os.environ["CUDA_VISIBLE_DEVICES"] = "-1"
os.environ["PYTHONHASHSEED"] = "0"

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np

plt.rcParams.update({
    "figure.dpi": 80,
    "savefig.dpi": 80,
    "savefig.bbox": None,
    "figure.autolayout": False,
    "axes.grid": True,
    "grid.alpha": 0.3,
    "font.size": 9,
})
plt.rcParams["font.family"] = "DejaVu Sans"

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
import pandas as pd
import random, gc

random.seed(42)
tf.random.set_seed(42)
print("TensorFlow:", tf.__version__, "| Keras:", keras.__version__)
print("Backend    :", keras.backend.backend())

I0000 00:00:1791728278.545253     455 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.


I0000 00:00:1791728280.556108     455 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.


TensorFlow: 2.21.0 | Keras: 3.15.1
Backend    : tensorflow


## 4. Carga de MNIST y partición estratificada

In [2]:
from sklearn.model_selection import train_test_split

SEED = 42
np.random.seed(SEED)

(x_train_full, y_train_full), (x_test_raw, y_test) = keras.datasets.mnist.load_data()

# División estratificada Train / Dev (80 / 20). Test queda intocable hasta el final.
x_train, x_dev, y_train, y_dev = train_test_split(
    x_train_full, y_train_full,
    test_size=0.20,
    random_state=SEED,
    stratify=y_train_full,
)

# Test se normaliza completo (31 MB); Train y Dev se normalizan por lotes para
# mantener el consumo de memoria del kernel bajo control.
x_test = x_test_raw.astype("float32") / 255.0


def normalizar_lote(imagenes):
    """Convierte un lote uint8 (n, 28, 28) en float32 (n, 28, 28) dentro de [0, 1].

    El aplanado a 784 características lo realiza la capa Flatten del modelo.
    """
    return imagenes.astype("float32") / 255.0


print("Train:", x_train.shape, "| Dev:", x_dev.shape, "| Test:", x_test.shape)

print()
print("Distribución de clases (conteo por dígito):")
print("Train completo:", np.bincount(y_train_full))
print("Train  :", np.bincount(y_train))
print("Dev    :", np.bincount(y_dev))
print("Test   :", np.bincount(y_test))

Train: (48000, 28, 28) | Dev: (12000, 28, 28) | Test: (10000, 28, 28)

Distribución de clases (conteo por dígito):
Train completo: [5923 6742 5958 6131 5842 5421 5918 6265 5851 5949]
Train  : [4738 5394 4766 4905 4674 4337 4734 5012 4681 4759]
Dev    : [1185 1348 1192 1226 1168 1084 1184 1253 1170 1190]
Test   : [ 980 1135 1032 1010  982  892  958 1028  974 1009]


### 4.1 Distribución de clases por conjunto

La partición estratificada (`stratify=y`) conserva la proporción de cada dígito en Train, Dev y Test.
Se verifica con una tabla que incluye el porcentaje: ninguna clase debe quedar sobrerrepresentada o
ausente en algún conjunto.

In [3]:
def distribucion_clases(y, nombre):
    """Tabla con conteo y porcentaje de cada dígito en un conjunto."""
    conteos = np.bincount(y, minlength=10)
    porcentajes = conteos / len(y) * 100
    tabla = pd.DataFrame({
        "digito": np.arange(10),
        "cantidad": conteos,
        "porcentaje": np.round(porcentajes, 2),
    })
    print("\n", nombre)
    display(tabla)
    print(tabla.to_string(index=False))
    return tabla


dist_train = distribucion_clases(y_train, "TRAIN")
dist_dev = distribucion_clases(y_dev, "DEV")
dist_test = distribucion_clases(y_test, "TEST")

print()
print("¿Los diez dígitos están presentes en los tres conjuntos?",
      bool((dist_train["cantidad"] > 0).all() and (dist_dev["cantidad"] > 0).all()
           and (dist_test["cantidad"] > 0).all()))
print("Desviación máxima de porcentaje entre Train y Dev:",
      round(float(np.abs(dist_train["porcentaje"] - dist_dev["porcentaje"]).max()), 2), "puntos")


 TRAIN


,digito,cantidad,porcentaje
0,0,4738,9.87
1,1,5394,11.24
2,2,4766,9.93
3,3,4905,10.22
4,4,4674,9.74
5,5,4337,9.04
6,6,4734,9.86
7,7,5012,10.44
8,8,4681,9.75
9,9,4759,9.91


 digito  cantidad  porcentaje
      0      4738        9.87
      1      5394       11.24
      2      4766        9.93
      3      4905       10.22
      4      4674        9.74
      5      4337        9.04
      6      4734        9.86
      7      5012       10.44
      8      4681        9.75
      9      4759        9.91

 DEV


,digito,cantidad,porcentaje
0,0,1185,9.88
1,1,1348,11.23
2,2,1192,9.93
3,3,1226,10.22
4,4,1168,9.73
5,5,1084,9.03
6,6,1184,9.87
7,7,1253,10.44
8,8,1170,9.75
9,9,1190,9.92


 digito  cantidad  porcentaje
      0      1185        9.88
      1      1348       11.23
      2      1192        9.93
      3      1226       10.22
      4      1168        9.73
      5      1084        9.03
      6      1184        9.87
      7      1253       10.44
      8      1170        9.75
      9      1190        9.92

 TEST


,digito,cantidad,porcentaje
0,0,980,9.80
1,1,1135,11.35
2,2,1032,10.32
3,3,1010,10.10
4,4,982,9.82
5,5,892,8.92
6,6,958,9.58
7,7,1028,10.28
8,8,974,9.74
9,9,1009,10.09


 digito  cantidad  porcentaje
      0       980        9.80
      1      1135       11.35
      2      1032       10.32
      3      1010       10.10
      4       982        9.82
      5       892        8.92
      6       958        9.58
      7      1028       10.28
      8       974        9.74
      9      1009       10.09

¿Los diez dígitos están presentes en los tres conjuntos? True
Desviación máxima de porcentaje entre Train y Dev: 0.01 puntos


### 4.2 Normalización de los píxeles

Dividir entre 255 lleva cada píxel al intervalo [0, 1]. Esto estabiliza el descenso de gradiente y
acelera la convergencia. Se comprueba el rango antes y después de normalizar un lote.

In [4]:
lote_crudo = x_train[:5]
lote_norm = normalizar_lote(lote_crudo)

print("Sin normalizar -> mínimo:", lote_crudo.min(), "máximo:", lote_crudo.max(),
      "| forma:", lote_crudo.shape)
print("Normalizado    -> mínimo:", lote_norm.min(), "máximo:", lote_norm.max(),
      "| forma:", lote_norm.shape, "(la capa Flatten lo convierte en 784 dentro del modelo)")
print()
print("La etiqueta no cambia con la normalización:",
      y_train[:5].tolist(), "->", y_train[:5].tolist())

Sin normalizar -> mínimo: 0 máximo: 255 | forma: (5, 28, 28)
Normalizado    -> mínimo: 0.0 máximo: 1.0 | forma: (5, 28, 28) (la capa Flatten lo convierte en 784 dentro del modelo)

La etiqueta no cambia con la normalización: [1, 3, 6, 8, 6] -> [1, 3, 6, 8, 6]


## 5. Modelo baseline 784 → 128 → 10

Arquitectura de referencia: una capa oculta densa de 128 neuronas ReLU y una capa de salida de 10
neuronas con Softmax. **No incluye capas convolucionales**, condición de la actividad. Este mismo
modelo se reutilizará en el Notebook 02 para comparar optimizadores en igualdad de condiciones.

In [5]:
def crear_modelo_baseline(units=128, optimizer=None):
    """Perceptrón multicapa 784 -> units -> 10 sin convoluciones."""
    modelo = keras.Sequential([
        layers.Input(shape=(28, 28)),
        layers.Flatten(),
        layers.Dense(units, activation="relu"),
        layers.Dense(10, activation="softmax"),
    ])
    if optimizer is None:
        optimizer = keras.optimizers.Adam(learning_rate=0.001)
    modelo.compile(
        optimizer=optimizer,
        loss="sparse_categorical_crossentropy",
        metrics=["accuracy"],
    )
    return modelo


tf.random.set_seed(42)
modelo_base = crear_modelo_baseline()
modelo_base.summary()

tipos = [capa.__class__.__name__ for capa in modelo_base.layers]
print()
print("Capas:", tipos)
print("¿Capas convolucionales?", any("Conv" in t for t in tipos),
      "| ¿Pooling?", any("Pooling" in t for t in tipos))
print("Parámetros entrenables:", f"{modelo_base.count_params():,}")

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ flatten (Flatten)               │ (None, 784)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 128)            │       100,480 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 10)             │         1,290 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 101,770 (397.54 KB)

 Trainable params: 101,770 (397.54 KB)

 Non-trainable params: 0 (0.00 B)


Capas: ['Flatten', 'Dense', 'Dense']
¿Capas convolucionales? False | ¿Pooling? False
Parámetros entrenables: 101,770


## 6. Entrenamiento con Dev explícito (5 épocas)

El ciclo se escribe de forma explícita con `tf.GradientTape` (forward → loss → backward → update),
procesando lotes de 128 imágenes. Es matemáticamente idéntico a `model.fit()`, pero convierte y
normaliza únicamente el lote en curso, lo que mantiene plano el consumo de memoria del kernel. Al
final de cada época se evalúa sobre **Dev**, que no participa en la actualización de pesos.

En Colab con memoria suficiente, la versión equivalente de alto nivel es:

```python
historial = modelo_base.fit(x_train_norm, y_train,
                            validation_data=(x_dev_norm, y_dev),
                            epochs=5, batch_size=64)
```

In [6]:
perdida_fn = keras.losses.SparseCategoricalCrossentropy()
optimizador = keras.optimizers.Adam(learning_rate=0.001)
BATCH_SIZE = 128
EPOCHS = 5
N_DEV = len(x_dev)


@tf.function
def paso_entrenamiento(xb, yb):
    """Un paso completo: forward -> loss -> backpropagation -> actualización."""
    with tf.GradientTape() as cinta:
        logits = modelo_base(xb, training=True)
        perdida = perdida_fn(yb, logits)
    gradientes = cinta.gradient(perdida, modelo_base.trainable_variables)
    optimizador.apply_gradients(zip(gradientes, modelo_base.trainable_variables))
    return perdida, tf.argmax(logits, axis=1)


def evaluar(x_datos, y_datos, tam_lote=1000):
    """Pérdida y exactitud por lotes, sin materializar todas las predicciones."""
    perdida_total, aciertos, total = 0.0, 0, 0
    for inicio in range(0, len(x_datos), tam_lote):
        xb = x_datos[inicio:inicio + tam_lote]
        yb = y_datos[inicio:inicio + tam_lote]
        xb = normalizar_lote(xb) if xb.dtype == np.uint8 else xb
        probs = modelo_base(xb, training=False).numpy()
        perdida_total += float(perdida_fn(yb, probs)) * len(yb)
        aciertos += int((probs.argmax(axis=1) == yb).sum())
        total += len(yb)
    return perdida_total / total, aciertos / total


indices_train = np.arange(len(x_train))
historial = {"loss": [], "accuracy": [], "val_loss": [], "val_accuracy": []}

for epoch in range(EPOCHS):
    np.random.shuffle(indices_train)
    suma_loss, aciertos, vistos = 0.0, 0, 0

    for inicio in range(0, len(indices_train), BATCH_SIZE):
        idx = indices_train[inicio:inicio + BATCH_SIZE]
        xb = normalizar_lote(x_train[idx])
        yb = y_train[idx]
        perdida, pred = paso_entrenamiento(xb, yb)
        suma_loss += float(perdida) * len(idx)
        aciertos += int((pred.numpy() == yb).sum())
        vistos += len(idx)

    val_loss, val_acc = evaluar(x_dev, y_dev)
    historial["loss"].append(suma_loss / vistos)
    historial["accuracy"].append(aciertos / vistos)
    historial["val_loss"].append(val_loss)
    historial["val_accuracy"].append(val_acc)
    gc.collect()

tabla_hist = pd.DataFrame({
    "epoca": range(1, EPOCHS + 1),
    "train_loss": np.round(historial["loss"], 4),
    "train_acc": np.round(historial["accuracy"], 4),
    "dev_loss": np.round(historial["val_loss"], 4),
    "dev_acc": np.round(historial["val_accuracy"], 4),
})
print("=== HISTORIAL DEL BASELINE ===")
display(tabla_hist)
print(tabla_hist.to_string(index=False))

=== HISTORIAL DEL BASELINE ===


,epoca,train_loss,train_acc,dev_loss,dev_acc
0,1,0.3881,0.8925,0.2196,0.9383
1,2,0.1781,0.9493,0.1616,0.9545
2,3,0.1283,0.9631,0.1320,0.9635
3,4,0.0984,0.9719,0.1175,0.9651
4,5,0.0796,0.9773,0.1058,0.9685


 epoca  train_loss  train_acc  dev_loss  dev_acc
     1      0.3881     0.8925    0.2196   0.9383
     2      0.1781     0.9493    0.1616   0.9545
     3      0.1283     0.9631    0.1320   0.9635
     4      0.0984     0.9719    0.1175   0.9651
     5      0.0796     0.9773    0.1058   0.9685


In [7]:
fig, axes = plt.subplots(1, 2, figsize=(10.5, 3.6))
epocas = range(1, EPOCHS + 1)

axes[0].plot(epocas, historial["loss"], marker="o", label="Train", color="#c0392b", lw=2)
axes[0].plot(epocas, historial["val_loss"], marker="s", label="Dev", color="#2980b9", lw=2, ls="--")
axes[0].set_title("Figura 1. Baseline - Loss")
axes[0].set_xlabel("Época"); axes[0].set_ylabel("Loss"); axes[0].legend(fontsize=8)

axes[1].plot(epocas, historial["accuracy"], marker="o", label="Train", color="#27ae60", lw=2)
axes[1].plot(epocas, historial["val_accuracy"], marker="s", label="Dev", color="#8e44ad", lw=2, ls="--")
axes[1].set_ylim(0.88, 1.0)
axes[1].set_title("Figura 2. Baseline - Accuracy")
axes[1].set_xlabel("Época"); axes[1].set_ylabel("Accuracy"); axes[1].legend(fontsize=8)

plt.tight_layout(); plt.savefig("figIII_01_baseline_curvas.png"); plt.show()
plt.close(fig); _ = gc.collect()

## 7. Lectura de las curvas

- **Train loss** disminuye de forma sostenida y **Dev loss** la acompaña sin divergencia marcada en
  5 épocas: todavía no hay overfitting apreciable con esta capacidad y este horizonte.
- Si en épocas posteriores Dev loss comenzara a subir mientras Train loss baja, esa separación sería
  la firma del overfitting; el Notebook 03 la provoca deliberadamente para estudiarla.
- La exactitud de Dev al final del entrenamiento es el número de referencia contra el cual se
  compararán optimizadores (Notebook 02) y regularizaciones (Notebook 03).

**Verificación de sanidad:** la pérdida de la época 1 debe ser mucho menor que −log(1/10) = 2.3026
(modelo sin entrenar) y la exactitud de Dev debe superar holgadamente el 10 % del azar.

In [8]:
print("Referencia sin entrenar  -ln(1/10) =", round(-np.log(0.1), 4))
print("Loss Dev época 1                    =", round(historial["val_loss"][0], 4))
print("Loss Dev época final                =", round(historial["val_loss"][-1], 4))
print("Accuracy Dev época final            =", round(historial["val_accuracy"][-1], 4))
print()
print("¿El modelo supera el azar (0.10)?", historial["val_accuracy"][-1] > 0.10)
print("¿La pérdida disminuyó respecto a la referencia?",
      historial["val_loss"][-1] < -np.log(0.1))

Referencia sin entrenar  -ln(1/10) = 2.3026
Loss Dev época 1                    = 0.2196
Loss Dev época final                = 0.1058
Accuracy Dev época final            = 0.9685

¿El modelo supera el azar (0.10)? True
¿La pérdida disminuyó respecto a la referencia? True


## 8. Conclusiones del notebook

1. La división **Train / Dev / Test** con funciones separadas es la condición metodológica mínima
   para poder afirmar algo sobre generalización.
2. La estratificación garantiza que las tres particiones tengan la misma composición de clases; sin
   ella, las métricas por clase serían incomparables.
3. Normalizar a [0, 1] es parte del modelo, no un detalle: cambia la velocidad y la estabilidad del
   descenso de gradiente.
4. El baseline queda fijado como punto de comparación: cualquier mejora futura debe medirse contra
   estos números con los mismos datos y la misma semilla.

## 9. Referencias

- Gulli, A., Kapoor, A., Pal, S., & Ciraci, S. (2023). *Deep Learning with TensorFlow and Keras* (3rd ed.). Packt Publishing.
- LeCun, Y., Cortes, C., & Burges, C. J. C. (2010). *MNIST handwritten digit database*. http://yann.lecun.com/exdb/mnist/
- Google (2026). *Google Colaboratory*. https://colab.research.google.com/